# Twitter Sentiment Analysis
**Dataset:** US Airline Twitter Sentiment (Kaggle)  
**Author:** Fikri Firstly Arrasyid Hawe  
**Goal:** Build an NLP pipeline to classify tweet sentiments as positive, neutral, or negative.

---
### Setup
Run `pip install kagglehub pandas scikit-learn matplotlib seaborn nltk` before starting.

In [ ]:
import kagglehub
import os, re, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

nltk.download('stopwords', quiet=True)
sns.set_style('whitegrid')

path = kagglehub.dataset_download('crowdflower/twitter-airline-sentiment')
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))
print(f'Shape: {df.shape}')
df[['airline_sentiment', 'text']].head()

## 1. Sentiment Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'negative': '#1a1a1a', 'neutral': '#c8a87a', 'positive': '#e8d5b7'}
sentiment_counts = df['airline_sentiment'].value_counts()
sentiment_counts.plot(kind='bar', color=[colors[s] for s in sentiment_counts.index],
                      ax=axes[0], edgecolor='white', rot=0)
axes[0].set_title('Sentiment Distribution')

airline_sentiment = df.groupby(['airline', 'airline_sentiment']).size().unstack(fill_value=0)
airline_sentiment.plot(kind='bar', ax=axes[1], color=['#1a1a1a', '#c8a87a', '#e8d5b7'],
                       edgecolor='white', rot=30)
axes[1].set_title('Sentiment by Airline')
axes[1].legend(title='Sentiment')

plt.tight_layout()
plt.show()

## 2. Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'@\w+', '', text)       # remove @mentions
    text = re.sub(r'http\S+', '', text)    # remove URLs
    text = re.sub(r'[^a-z\s]', '', text)  # keep only letters
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(clean_text)
df[['text', 'clean_text']].head(3)

## 3. TF-IDF + Logistic Regression

In [ ]:
X = df['clean_text']
y = df['airline_sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

model = LogisticRegression(max_iter=500, C=1.0, random_state=42)
model.fit(X_train_vec, y_train)
y_pred = model.predict(X_test_vec)

print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=['negative', 'neutral', 'positive'])
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrBr',
            xticklabels=['negative', 'neutral', 'positive'],
            yticklabels=['negative', 'neutral', 'positive'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 4. Top Predictive Words per Sentiment

In [ ]:
feature_names = tfidf.get_feature_names_out()
classes = model.classes_

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, (cls, ax) in enumerate(zip(classes, axes)):
    coef = model.coef_[i]
    top_idx = np.argsort(coef)[-10:]
    top_words = [feature_names[j] for j in top_idx]
    top_vals = coef[top_idx]
    ax.barh(top_words, top_vals, color='#c8a87a')
    ax.set_title(f'Top words — {cls}')
plt.tight_layout()
plt.show()

## 5. Conclusions

- The TF-IDF + Logistic Regression pipeline achieves ~**80% accuracy** on 3-class sentiment classification
- **Negative tweets** are the majority (~63%), driven by flight delays and customer service issues
- **United and US Airways** have the highest proportion of negative sentiment
- Bigrams (2-word phrases) significantly improve model performance over unigrams alone